# Run 20 — Large model + Run-19 regularization (capacity re-test) — Colab T4 / TPU

Re-opens the capacity question **under controlled overfitting**. The earlier capacity A/B
(`src/ab_capacity_test.py`, Exp 25) was run with the *light* recipe, where the 7M model already
over-fits (train → 80%, val collapses). Under that recipe S→M gave +2 pp but M→L looked pointless.
**But capacity and regularization are coupled** — a bigger net can only exploit its capacity if it is
stopped from memorizing. So this notebook tests **L (14.8M) with the exact Run 19 recipe**
(MixUp α0.2 from ep15, wd 3e-3, drop_path 0.05), making it a fair, single-variable comparison against
Run 19's M.

## Read the result (compare vs. Run 19 / `02_tpu_train.ipynb`, same recipe, M = 7M)

| Outcome | Meaning | Action |
|---------|---------|--------|
| **L_val ≈ M_val (±1 pp)** | capacity is saturated even with reg | stay on M (cheaper) |
| **L_val > M_val (≥2 pp)** | capacity *does* help once overfit is controlled | scale up; this is a real lever |
| **L runs a positive gap (train ≫ val)** | L over-fits despite reg | raise `drop_path`→0.1 / wd→5e-3 and retry |

## Config

`channels=(64,128,256,448)`, `depths=(2,3,4,2)` (**14.81M**, ~2× M). **Recipe identical to Run 19**:
MixUp α0.2 from ep15 (no CutMix), wd 3e-3, drop_path 0.05, dropout 0.1, SpecAugment 8/4,
EMA 0.995+reset@3, OneCycleLR max_lr 2.5e-3, 70 epochs, G channel 128×64. `N_FOLDS=1` (single-split
probe vs M; switch to 3 only if L clearly wins).

## How to run
1. *Runtime → T4 GPU* (or TPU). 2. Kaggle creds in Secrets (🔑). 3. Run all. Compare best val **and**
the `gap` trajectory directly against Run 19's M.

In [ ]:
# === 1. Dependencies (torch+CUDA is preinstalled on Colab GPU runtimes) ===
!pip install -q kagglehub matplotlib

In [ ]:
# === 2. Device ===
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
print('torch', torch.__version__, '| device:', device,
      '|', torch.cuda.get_device_name(0) if use_amp else 'CPU')

## 3. Get the competition data

The notebook uses `kagglehub`, which needs your Kaggle API token. Easiest on Colab:
**Settings (left sidebar 🔑) → add two secrets** `KAGGLE_USERNAME` and `KAGGLE_KEY`
(from your `kaggle.json`), then run the cell. Fallback: upload `kaggle.json` when prompted.

In [ ]:
# === 3. Download data ===
import os
from pathlib import Path

# Pull Kaggle credentials from Colab secrets if present
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except Exception:
    pass

# Fallback: prompt for kaggle.json upload
if not os.environ.get('KAGGLE_KEY'):
    try:
        from google.colab import files
        print('Upload your kaggle.json:')
        up = files.upload()
        import json as _json
        kj = _json.loads(next(iter(up.values())))
        os.environ['KAGGLE_USERNAME'] = kj['username']
        os.environ['KAGGLE_KEY'] = kj['key']
    except Exception as e:
        print('No credentials set:', e)

import kagglehub
raw = Path(kagglehub.competition_download('signal-object-detection'))
DATA_DIR = raw
print('Data at:', DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))

## 4. Decode every image to a single **G channel** at `128×64`

Exp 21 (see `README.md`) settled the input question with a controlled A/B: **viridis-inversion was
−8.4 pp worse** than the raw G channel, and **full RGB tied** with G. The spectrograms are a colormap
of a single scalar field, so the G channel carries all the signal and is the cheapest representation.
We decode each PNG to its G channel, resize to `128×64`, and cache the tensors in RAM.

**Resolution** was also A/B-tested (Exp 23): 128×64 vs 128×128 vs 224×224 gave 58.61 / 58.77 / 59.26% — +0.65pp at ~12× compute for 224. Not a lever, so we keep the cheap **128×64**.

In [ ]:
# === 4. Decode all images to single G channel (cached) ===
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

IMG_H, IMG_W = 128, 64

def decode_image(path):
    g = np.array(Image.open(path))[:, :, 1].astype(np.float32) / 255.0   # G channel of viridis PNG
    t = torch.from_numpy(g)[None, None]                                   # [1,1,H,W]
    t = F.interpolate(t, size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)
    return t[0]                                                           # [1,IMG_H,IMG_W]

def decode_split(csv, img_dir, has_label=True):
    df = pd.read_csv(csv)
    X = torch.empty(len(df), 1, IMG_H, IMG_W)
    for i, row in enumerate(df.itertuples(index=False)):
        X[i] = decode_image(Path(img_dir) / row.id)
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(df)}')
    y = torch.tensor((df['label'].values - 1)) if has_label else None
    return X, y, df['id'].tolist()

from pathlib import Path
cache = Path('/content/decoded_g.pt')
if cache.exists():
    blob = torch.load(cache)
    Xtr, ytr, Xte, test_ids = blob['Xtr'], blob['ytr'], blob['Xte'], blob['test_ids']
    print('loaded cache')
else:
    print('decoding train...');  Xtr, ytr, _       = decode_split(DATA_DIR/'train.csv', DATA_DIR/'train', True)
    print('decoding test...');   Xte, _, test_ids  = decode_split(DATA_DIR/'test.csv',  DATA_DIR/'test',  False)
    torch.save({'Xtr':Xtr,'ytr':ytr,'Xte':Xte,'test_ids':test_ids}, cache)
print('train', tuple(Xtr.shape), 'test', tuple(Xte.shape))

MEAN = Xtr.mean().item()
STD  = Xtr.std().item()
print(f'norm  mean={MEAN:.4f}  std={STD:.4f}')
print('class balance:', torch.bincount(ytr).tolist())

## 5. Augmentation (spectrogram-valid only) + MixUp / CutMix

Per-sample: brightness/contrast jitter, light Gaussian noise, SpecAugment frequency &
time masking. **No flips or rotations** — the time/frequency axes are semantic. MixUp and
CutMix are applied at the batch level after a warmup (MixUp from epoch 1 stalls a
from-scratch net — README Exp 14).

In [ ]:
# === 5. Dataset + augmentation (light — fold0 was under-fit, so we ease off) ===
from torch.utils.data import Dataset

class SpecDataset(Dataset):
    def __init__(self, X, y, train=True, fmask=8, tmask=4, noise=0.02):
        self.X, self.y, self.train = X, y, train
        self.fmask, self.tmask, self.noise = fmask, tmask, noise
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].clone()                       # [1,H,W] in [0,1]
        if self.train:
            c = 0.8 + 0.4 * torch.rand(1).item()    # contrast 0.8..1.2
            b = (torch.rand(1).item() - 0.5) * 0.1  # brightness -0.05..0.05
            x = ((x - 0.5) * c + 0.5 + b).clamp_(0, 1)
            if self.noise:
                x.add_(torch.randn_like(x) * self.noise).clamp_(0, 1)
            H, W = x.shape[1], x.shape[2]
            f = int(torch.randint(0, self.fmask + 1, (1,)).item())
            f0 = int(torch.randint(0, max(1, H - f), (1,)).item()); x[:, f0:f0+f, :] = 0.0
            t = int(torch.randint(0, self.tmask + 1, (1,)).item())
            t0 = int(torch.randint(0, max(1, W - t), (1,)).item()); x[:, :, t0:t0+t] = 0.0
        x = (x - MEAN) / STD
        return x, int(self.y[i])

def mixup_cutmix(x, y, alpha=0.3, cutmix_prob=0.5):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    if np.random.rand() < cutmix_prob:                       # CutMix: paste a time-freq patch
        H, W = x.size(2), x.size(3)
        rh, rw = int(H * np.sqrt(1 - lam)), int(W * np.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - rh // 2, 0), min(cy + rh // 2, H)
        x1, x2 = max(cx - rw // 2, 0), min(cx + rw // 2, W)
        x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
        lam = 1 - ((y2 - y1) * (x2 - x1) / (H * W))
    else:                                                    # MixUp: linear blend
        x = lam * x + (1 - lam) * x[idx]
    return x, y, y[idx], lam

In [ ]:
# === 6. Model — from PyTorch primitives only ===
import torch.nn as nn

class DropPath(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0.0 or not self.training: return x
        keep = 1 - self.p
        mask = torch.empty((x.size(0),) + (1,)*(x.ndim-1), dtype=x.dtype, device=x.device).bernoulli_(keep)
        return x / keep * mask

class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__(); s = max(1, c // r); self.c = c
        self.fc = nn.Sequential(nn.Linear(c, s, bias=False), nn.SiLU(),
                                nn.Linear(s, c, bias=False), nn.Sigmoid())
    def forward(self, x):
        w = self.fc(x.mean((2, 3))).view(x.size(0), self.c, 1, 1)
        return x * w

class BlurPool(nn.Module):
    # Anti-aliased stride-2 downsample with a fixed depthwise binomial kernel.
    def __init__(self, c):
        super().__init__(); self.c = c
        k = torch.tensor([1., 2., 1.]); k = (k[:, None] * k[None, :]); k = k / k.sum()
        self.register_buffer('k', k[None, None].repeat(c, 1, 1, 1))
    def forward(self, x):
        return F.conv2d(x, self.k, stride=2, padding=1, groups=self.c)

class ConvBNAct(nn.Module):
    def __init__(self, i, o, k=3, s=1, p=1, g=1):
        super().__init__()
        self.conv = nn.Conv2d(i, o, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(o); self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class ResBlock(nn.Module):
    def __init__(self, c, dp=0.0):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(c)
        self.se = SEBlock(c); self.act = nn.SiLU(inplace=True); self.dp = DropPath(dp)
    def forward(self, x):
        out = self.act(self.b1(self.c1(x)))
        out = self.se(self.b2(self.c2(out)))
        return self.act(x + self.dp(out))

class Down(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.conv = ConvBNAct(i, o, 3, 1, 1); self.pool = BlurPool(o)
    def forward(self, x): return self.pool(self.conv(x))

class SignalNetV2(nn.Module):
    def __init__(self, num_classes=5, channels=(48, 96, 192, 320),
                 depths=(2, 2, 3, 2), dropout_p=0.4, drop_path=0.15):
        super().__init__()
        self.stem = ConvBNAct(1, channels[0], 3, 1, 1)
        total = sum(depths); dpr = [drop_path * i / max(1, total - 1) for i in range(total)]
        stages, prev, bi = [], channels[0], 0
        for c, d in zip(channels, depths):
            layers = [Down(prev, c)]
            for _ in range(d): layers.append(ResBlock(c, dpr[bi])); bi += 1
            stages.append(nn.Sequential(*layers)); prev = c
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(prev, prev), nn.SiLU(inplace=True),
                                  nn.Dropout(dropout_p), nn.Linear(prev, num_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
        for m in self.modules():               # zero-init last BN in each residual block
            if isinstance(m, ResBlock): nn.init.zeros_(m.b2.weight)
    def forward(self, x): return self.head(self.stages(self.stem(x)))

_m = SignalNetV2(channels=(64,128,256,448), depths=(2,3,4,2)); _n = sum(p.numel() for p in _m.parameters())
print(f'SignalNetV2 params: {_n:,} ({_n/1e6:.2f}M)')

In [ ]:
# === 7. EMA (exponential moving average of weights) ===
import copy

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            if s.dtype.is_floating_point: s.mul_(self.decay).add_(m, alpha=1 - self.decay)
            else: s.copy_(m)
    def module(self): return self.shadow

In [ ]:
# === 8. Train one fold (CUDA + AMP) — Run 18 recipe "let it fit" ===
CONFIG = dict(
    epochs=70, batch_size=256, max_lr=2.5e-3, weight_decay=3e-3,
    label_smoothing=0.1, mixup_alpha=0.2, mixup_warmup=15,    # Run 19 recipe: MixUp ON from ep15 (no CutMix)
    ema_decay=0.995, ema_reset_epoch=3, pct_start=0.25,
    patience=60,                                              # = epochs -> no early stop (EMA peaks at the end)
    channels=(64, 128, 256, 448), depths=(2, 3, 4, 2),   # LARGE ~14.8M (vs M ~7.07M)
    dropout_p=0.1, drop_path=0.05,                           # Run 19 recipe (mild structural reg)
)

from torch.utils.data import DataLoader

def evaluate(model, X, y, bs=512):
    model.eval(); correct = 0
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        for s in range(0, len(X), bs):
            xb = ((X[s:s+bs] - MEAN) / STD).to(device)
            pred = model(xb).float().argmax(1).cpu()
            correct += (pred == y[s:s+bs]).sum().item()
    return 100.0 * correct / len(X)

def train_fold(tr_idx, va_idx, cfg, tag='fold0', seed=42):
    torch.manual_seed(seed)                                  # per-fold seed -> ensemble diversity
    ds = SpecDataset(Xtr[tr_idx], ytr[tr_idx], train=True)
    loader = DataLoader(ds, batch_size=cfg['batch_size'], shuffle=True,
                        num_workers=2, drop_last=True, persistent_workers=True, pin_memory=use_amp)
    Xva, yva = Xtr[va_idx], ytr[va_idx]

    model = SignalNetV2(5, cfg['channels'], cfg['depths'],
                        cfg['dropout_p'], cfg['drop_path']).to(device)
    ema = EMA(model, cfg['ema_decay'])
    crit = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['max_lr'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    steps = len(loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['max_lr'],
                total_steps=cfg['epochs'] * steps, pct_start=cfg['pct_start'])

    best, best_state, bad = 0.0, None, 0
    for ep in range(cfg['epochs']):
        if ep == cfg['ema_reset_epoch']:        # reset EMA shadow to settled weights (kills init bias)
            ema = EMA(model, cfg['ema_decay'])
        model.train()
        tr_correct = torch.zeros((), device=device); tr_total = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                if ep >= cfg['mixup_warmup']:
                    xm_, ya, yb, lam = mixup_cutmix(x, y, cfg['mixup_alpha'], cutmix_prob=0.0)
                    out = model(xm_); loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb)
                else:
                    out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            with torch.no_grad():
                tr_correct += (out.float().argmax(1) == y).sum()
            tr_total += y.size(0)
        train_acc = 100.0 * tr_correct.item() / tr_total
        raw_acc = evaluate(model, Xva, yva)
        ema_acc = evaluate(ema.module(), Xva, yva)
        acc = max(raw_acc, ema_acc); src = 'EMA' if ema_acc >= raw_acc else 'raw'
        gap = train_acc - acc                   # negative => under-fit; large positive => over-fit
        print(f'[{tag}] ep {ep+1}/{cfg["epochs"]}  train {train_acc:.2f}  raw {raw_acc:.2f}  '
              f'ema {ema_acc:.2f}  -> val {acc:.2f} ({src})  gap {gap:+.1f}  '
              f'lr {opt.param_groups[0]["lr"]:.2e}', flush=True)
        if acc > best:
            best, bad = acc, 0
            chosen = ema.module() if ema_acc >= raw_acc else model
            best_state = {k: v.detach().cpu().clone() for k, v in chosen.state_dict().items()}
            torch.save(best_state, f'/content/best_{tag}.pt')
        else:
            bad += 1
            if bad >= cfg['patience']:
                print(f'[{tag}] early stop @ ep {ep+1}'); break
    print(f'[{tag}] BEST {best:.2f}%')
    return best, best_state

In [ ]:
# === 9. K-fold training (set N_FOLDS=1 to calibrate fast, 3 for the full ensemble) ===
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 1   # capacity probe: single split vs M; set 3 only if L wins
_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)   # always 3 splits; take first N_FOLDS
_splits = list(_skf.split(np.zeros(len(ytr)), ytr.numpy()))[:N_FOLDS]
fold_states, fold_accs = [], []
for f, (tr, va) in enumerate(_splits):
    print(f'\n===== FOLD {f+1}/{N_FOLDS} =====')
    acc, state = train_fold(tr, va, CONFIG, tag=f'L_fold{f}', seed=42 + f)
    fold_accs.append(acc); fold_states.append(state)
print('\nCV mean val acc: %.2f%%  (Kaggle est. %.2f%%)' %
      (np.mean(fold_accs), np.mean(fold_accs) + 2.7))

## 10. Inference — K-fold ensemble + TTA → `submission.csv`

Averages softmax over all folds and a few **valid** test-time augmentations (identity +
brightness/contrast variants). No geometric flips.

In [ ]:
# === 10. Ensemble + TTA inference ===
def tta_views(xb):
    yield (xb - MEAN) / STD
    for c, b in [(0.9, 0.0), (1.1, 0.0), (1.0, 0.05), (1.0, -0.05)]:
        yield (((xb - 0.5) * c + 0.5 + b).clamp(0, 1) - MEAN) / STD

probs = torch.zeros(len(Xte), 5)
for state in fold_states:
    model = SignalNetV2(5, CONFIG['channels'], CONFIG['depths'],
                        CONFIG['dropout_p'], CONFIG['drop_path']).to(device)
    model.load_state_dict(state); model.eval()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        for s in range(0, len(Xte), 512):
            raw = Xte[s:s+512]
            acc = torch.zeros(raw.size(0), 5)
            for v in tta_views(raw):
                acc += F.softmax(model(v.to(device)).float(), 1).cpu()
            probs[s:s+512] += acc / 5
pred = probs.argmax(1).numpy() + 1   # back to 1-indexed

sub = pd.DataFrame({'id': test_ids, 'label': pred})
sub.to_csv('/content/submission.csv', index=False)
print(sub['label'].value_counts().sort_index())
print(sub.head())
try:
    from google.colab import files; files.download('/content/submission.csv')
except Exception: pass

## 11. Reading the run & what to change next

**The `gap = train − val` column is the steering signal.** Decide from it, not from vibes:

- **`gap` negative (val > train), val low** → still **under-fit** (where fold0 sat at ≈ −8 pp).
  Reduce regularization further (`noise→0`, SpecAugment `fmask=4/tmask=2` or off), raise `max_lr`
  toward 3e-3, or add epochs. Do this *before* anything else.
- **`gap` ≈ 0, val plateaus** → you have hit the **capacity / data ceiling**. Now widen the model
  (`channels=(64,128,256,448)`) or deepen it (`depths=(2,3,4,2)`); this is the next real lever the
  A/Bs point to.
- **`gap` large positive (train ≫ val)** → **over-fit**. *Then* (and only then) add regularization
  back one knob at a time: MixUp `mixup_warmup=15, mixup_alpha=0.2`, then `drop_path=0.1`,
  `dropout_p=0.3`, re-checking the gap after each.

**Other levers** (after the recipe is healthy): `N_FOLDS=5` (stronger ensemble, ~+1%); train a second
*different* architecture and average it in.

**Persistence** — Colab can reset; copy checkpoints to Drive:
`from google.colab import drive; drive.mount('/content/drive')` then `!cp /content/best_*.pt /content/decoded_g.pt /content/drive/MyDrive/`.

**Log every run** (change → val acc → Kaggle) in `README.md` so the experiment history stays intact.